# Merchant Risk Analytics

## Phase 1 – Data Profiling

### Objective

The purpose of this notebook is to profile the PaySim transaction dataset before feature engineering and risk modeling.

The analysis focuses on:

- Understanding the structure of the dataset
- Evaluating data quality
- Identifying business entities
- Exploring transaction behavior
- Establishing a baseline for merchant risk analytics

In [5]:
import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import text

## 2. Database Connection

In [6]:
project_root = Path.cwd().parent
sys.path.append(str(project_root))

from config.database import engine

with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database();"))
    print(f"Connected to database: {result.fetchone()[0]}")

Connected to database: postgres


## 3. Dataset Preview

In [7]:
df_preview = pd.read_sql("""
SELECT *
FROM public.staging_paysim_ledger
LIMIT 10;
""", engine)

df_preview

,step,type,amount,nameorig,oldbalanceorg,newbalanceorig,namedest,oldbalancedest,newbalancedest,isfraud,isflaggedfraud
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.0,0.00,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.0,0.00,0,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.0,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.0,0.00,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.0,0.00,0,0
5,1,PAYMENT,7817.71,C90045638,53860.00,46042.29,M573487274,0.0,0.00,0,0
6,1,PAYMENT,7107.77,C154988899,183195.00,176087.23,M408069119,0.0,0.00,0,0
7,1,PAYMENT,7861.64,C1912850431,176087.23,168225.59,M633326333,0.0,0.00,0,0
8,1,PAYMENT,4024.36,C1265012928,2671.00,0.00,M1176932104,0.0,0.00,0,0
9,1,DEBIT,5337.77,C712410124,41720.00,36382.23,C195600860,41898.0,40348.79,0,0


### Observation

The dataset loaded successfully from PostgreSQL.

Each record represents a single financial transaction containing:

- Transaction details (type, amount)
- Customer and merchant identifiers
- Account balances before and after the transaction
- Fraud indicators (`isfraud` and `isflaggedfraud`)

The dataset is ready for detailed profiling.

## 4. Dataset Dimensions

In [8]:
total_rows = pd.read_sql("""
SELECT COUNT(*) AS total_rows
FROM public.staging_paysim_ledger;
""", engine)

total_columns = pd.read_sql("""
SELECT COUNT(*) AS total_columns
FROM information_schema.columns
WHERE table_schema = 'public'
  AND table_name = 'staging_paysim_ledger';
""", engine)

print(f"Total Rows    : {total_rows.iloc[0, 0]:,}")
print(f"Total Columns : {total_columns.iloc[0, 0]}")

Total Rows    : 6,362,620
Total Columns : 11


## 5. Schema Inspection

In [9]:
schema_df = pd.read_sql("""
SELECT
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'public'
  AND table_name = 'staging_paysim_ledger'
ORDER BY ordinal_position;
""", engine)

schema_df

,column_name,data_type
0,step,integer
1,type,character varying
2,amount,numeric
3,nameorig,character varying
4,oldbalanceorg,numeric
5,newbalanceorig,numeric
6,namedest,character varying
7,oldbalancedest,numeric
8,newbalancedest,numeric
9,isfraud,integer


## 6. Data Quality Assessment

In [10]:
missing_values = pd.read_sql("""
SELECT
    COUNT(*) - COUNT(step) AS step,
    COUNT(*) - COUNT(type) AS type,
    COUNT(*) - COUNT(amount) AS amount,
    COUNT(*) - COUNT(nameorig) AS nameorig,
    COUNT(*) - COUNT(oldbalanceorg) AS oldbalanceorg,
    COUNT(*) - COUNT(newbalanceorig) AS newbalanceorig,
    COUNT(*) - COUNT(namedest) AS namedest,
    COUNT(*) - COUNT(oldbalancedest) AS oldbalancedest,
    COUNT(*) - COUNT(newbalancedest) AS newbalancedest,
    COUNT(*) - COUNT(isfraud) AS isfraud,
    COUNT(*) - COUNT(isflaggedfraud) AS isflaggedfraud
FROM public.staging_paysim_ledger;
""", engine)

missing_values.T.rename(columns={0: "Missing Values"})

,Missing Values
step,0
type,0
amount,0
nameorig,0
oldbalanceorg,0
newbalanceorig,0
namedest,0
oldbalancedest,0
newbalancedest,0
isfraud,0


## 7. Duplicate Record Assessment

In [11]:
duplicate_records = pd.read_sql("""
SELECT COUNT(*) AS duplicate_records
FROM (
    SELECT *,
           COUNT(*) OVER (
               PARTITION BY
                   step, type, amount, nameorig,
                   oldbalanceorg, newbalanceorig,
                   namedest, oldbalancedest,
                   newbalancedest, isfraud, isflaggedfraud
           ) AS cnt
    FROM public.staging_paysim_ledger
) t
WHERE cnt > 1;
""", engine)

duplicate_records

,duplicate_records
0,0


## 8. Transaction Type Distribution

In [12]:
transaction_types = pd.read_sql("""
SELECT
    type AS transaction_type,
    COUNT(*) AS transaction_count,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM public.staging_paysim_ledger
GROUP BY type
ORDER BY transaction_count DESC;
""", engine)

transaction_types

,transaction_type,transaction_count,percentage
0,CASH_OUT,2237500,35.17
1,PAYMENT,2151495,33.81
2,CASH_IN,1399284,21.99
3,TRANSFER,532909,8.38
4,DEBIT,41432,0.65


## 8. Merchant Transaction Validation

Validate that merchant transactions are represented by destination IDs beginning with `M` and determine the proportion of transactions relevant for merchant-level analytics.

In [13]:
merchant_validation = pd.read_sql("""
SELECT
    CASE
        WHEN LEFT(namedest,1)='M' THEN 'Merchant'
        ELSE 'Customer'
    END AS destination_type,
    COUNT(*) AS transaction_count,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM public.staging_paysim_ledger
GROUP BY destination_type
ORDER BY transaction_count DESC;
""", engine)

merchant_validation

,destination_type,transaction_count,percentage
0,Customer,4211125,66.19
1,Merchant,2151495,33.81


## 9. Fraud Distribution

Establish the baseline fraud rate within the transaction dataset. This benchmark will later be used to evaluate merchant-level behavioral risk metrics.

In [14]:
fraud_summary = pd.read_sql("""
SELECT
    COUNT(*) AS total_transactions,
    SUM(isfraud) AS fraudulent_transactions,
    ROUND(
        SUM(isfraud) * 100.0 / COUNT(*),
        4
    ) AS fraud_rate_percentage
FROM public.staging_paysim_ledger;
""", engine)

fraud_summary

,total_transactions,fraudulent_transactions,fraud_rate_percentage
0,6362620,8213,0.1291
